# Thiết kế bộ KPI cho Capital Bikeshare: north star, input, guardrail và scorecard 28 ngày

Bộ `bike-sharing-uci`, `day.csv`: 731 ngày (01/01/2011 tới 31/12/2012). `cnt` = `casual` (khách vãng lai, trả theo
lượt) + `registered` (thành viên, trả phí năm/tháng); `weathersit` 1 = quang, 2 = mây/sương, 3 = mưa/tuyết nhẹ,
4 = mưa to.

**Bối cảnh.** Ban giám đốc mới của hệ thống xe đạp nhận một dashboard 40 chỉ số và nói: *"Tôi không biết công ty đang
tốt lên hay xấu đi. Cho tôi **một** con số để cả công ty cùng kéo, vài con số mà từng team điều khiển được, và một
scorecard xem mỗi tháng."*

Bạn làm hai việc:

1. Viết `KPI_FRAMEWORK.md`: north star, input metrics, cặp leading/lagging, guardrails, metric tree, nhịp xem và chủ sở
   hữu, rủi ro "game" metric.
2. Viết `kpi_scorecard(day, end, window)`: scorecard 28 ngày so với **cùng 28 ngày năm trước, lệch 364 ngày** (364 = 52
   tuần, để mỗi ngày trong tuần so với đúng ngày đó: thứ Hai với thứ Hai).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm hoặc hằng. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/01-analyst-mindset/exercises/advanced/02-north-star-kpi-framework -q` (kiểm hàm scorecard và cấu trúc `KPI_FRAMEWORK.md`). Chất lượng bộ KPI chấm theo rubric trong lời giải.

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có, đọc day.csv
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

BIKE = ensure("bike-sharing-uci")
day = pd.read_csv(BIKE / "day.csv")
print(day.shape, day.dteday.min(), "→", day.dteday.max())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Scorecard 28 ngày so với năm trước

`kpi_scorecard(day, end, window=28) -> pd.DataFrame`:

- **Kỳ hiện tại**: các ngày `dteday` từ `end - (window - 1)` tới `end` (gồm cả hai đầu). `end` là chuỗi ngày như
  `"2012-10-31"`.
- **Kỳ so sánh**: cùng độ dài, kết thúc ở `end - 364 ngày`.
- Index là năm metric theo thứ tự `METRICS`; ba cột `current`, `previous`, `change_pct`
  (`current / previous - 1`, `NaN` khi `previous` bằng 0).

| Metric | Định nghĩa trên các ngày của kỳ |
| --- | --- |
| `rides` | tổng `cnt` |
| `registered_rides` | tổng `registered` |
| `rides_per_day` | trung bình `cnt` mỗi ngày |
| `registered_share` | `registered_rides / rides` |
| `bad_weather_days` | số ngày `weathersit >= 3` |

Scorecard cố ý có cả metric kết quả lẫn metric bối cảnh (thời tiết): người đọc cần biết "tăng" là nhờ công ty hay nhờ
trời.

In [ ]:
METRICS = ["rides", "registered_rides", "rides_per_day", "registered_share", "bad_weather_days"]


def kpi_scorecard(day: pd.DataFrame, end: str, window: int = 28) -> pd.DataFrame:
    """Index METRICS; cột current, previous (lệch 364 ngày), change_pct."""
    # TODO: lọc hai kỳ theo pd.to_datetime(dteday); tính năm metric mỗi kỳ; change_pct = current / previous - 1
    raise NotImplementedError

In [ ]:
# explore: scorecard của vài tháng cuối năm 2012
for end in ["2012-09-30", "2012-10-31", "2012-12-31"]:
    print(f"== 28 ngày tới {end}")
    print(kpi_scorecard(day, end).round(3).to_string())

In [ ]:
# explore: ứng viên north star - lượt thuê của thành viên theo tuần, so với 52 tuần trước
dates = pd.to_datetime(day["dteday"])
weekly = day.assign(week=dates.dt.to_period("W-SUN").dt.start_time).groupby("week")[["registered", "casual"]].sum()
weekly = weekly.iloc[1:-1]                      # bỏ tuần đầu và cuối không đủ 7 ngày
yoy = weekly["registered"] / weekly["registered"].shift(52) - 1
fig, (a, b) = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True)
a.plot(weekly.index, weekly["registered"] / 1e3, color="#4c78a8", label="thành viên")
a.plot(weekly.index, weekly["casual"] / 1e3, color="#f58518", label="vãng lai")
a.set_ylabel("nghìn lượt / tuần")
a.legend(frameon=False)
b.plot(yoy.index, yoy * 100, color="#4c78a8")
b.axhline(0, color="gray", lw=0.8)
b.set_ylabel("% so với 52 tuần trước")
for ax in (a, b):
    ax.spines[["top", "right"]].set_visible(False)
a.set_title("Lượt thuê của thành viên theo tuần: mức và tăng trưởng so cùng kỳ", loc="left")
fig.tight_layout()
plt.show()
print(yoy.dropna().describe().round(3))

## 2. Viết `KPI_FRAMEWORK.md`

Mở `starter/KPI_FRAMEWORK.md`, điền theo khung (300-900 từ, xoá mọi dòng `TODO`). Mục "Metric tree" có một sơ đồ
` ```mermaid `. Câu hỏi để tự kiểm:

- North star có thể hiện **giá trị khách nhận được** không, hay chỉ là doanh thu đổi tên? Nó có tăng được bằng cách
  làm hại khách không?
- Mỗi input metric có **một team** điều khiển được không (marketing, vận hành trạm, sản phẩm app, giá)?
- Mỗi KPI lagging có leading indicator đi kèm chưa? Leading đó đã được kiểm là dự báo được chưa (xem bài Basic 03)?
- Guardrail nào chặn việc "đạt KPI" bằng cách làm hại khách hoặc tài chính?
- Bộ dữ liệu này **không có** gì (doanh thu, số thành viên, số xe, trạm) và bạn cần thêm nguồn nào?